# Droplet rebound simulation (gauthier experiments) - Run 2D with heat equation

This notebook is used to run the actual simulations for the droplet rebound test case. The simulation will be restarted from a previously computed initial state, see `DropletReboundInit.ipynb`

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile()
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init()
   at Submission#21.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [ ]:
Console.WriteLine("Execution Date/time is " + DateTime.Now);

In [ ]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [ ]:
ExecutionQueues

index type DeploymentBaseDirectory DeployRuntime RuntimeLocation Name DotnetRuntime BatchInstructionDir AllowedDatabasesPaths AdditionalEnvironmentVars Username NumOfAdditionalServiceCores NumOfAdditionalServiceCoresMPISerial NumOfServiceCoresPerMPIprocess ServerName ComputeNodes DefaultJobPriority SingleNode Password SshClientExeToUse PrivateKeyFilePath AdditionalBatchCommands .. 0 BoSSS.Application.BoSSSpad.MiniBatchProcessorClient D:\local\binaries False <null> LocalPC dotnet <null> [ D:\local\ ] 1 BoSSS.Application.BoSSSpad.MsHPC2012Client \\dc3\userspace\smuda\hpccluster\binaries True win\amd64 FDY-WindowsHPC dotnet [ \\dc3\userspace\smuda\hpccluster\ ] [ ] FDY\smuda 0 0 0 DC3 <null> Normal True 2 BoSSS.Application.BoSSSpad.SlurmClient \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_deploy False linux/amd64-openmpi Lb2-specialPrj dotnet [ \\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases == /work/scratch/nu39gihu/bosss_databases ] nu39gihu lcluster17.hrz.tu-darmstadt.de <null> C:\Program Files\Git\usr\bin\ssh.exe \\dc3\userspace\smuda\.ssh\id_lichtb [ #SBATCH -C avx512, #SBATCH --mem-per-cpu=3800 ]

In [ ]:
var myBatch = ExecutionQueues[1];

if(userName.Equals(@"FDY\JenkinsCI"))
    myBatch = GetDefaultQueue();

myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletRebound_Gauthier_Heat", myBatch);

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setting up restart

Choose if this notebook should do a restart. 

In [ ]:
bool restartRun = false;

In [ ]:
OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier_Heat");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\DropletRebound_Gauthier");
//OpenOrCreateDatabase(@"D:\local\DropletRebound_Gauthier");

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_LineSearch_restart1*	02/20/2025 10:07:39	aa479395...
#1: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5_LineSearch_restart1*	02/20/2025 10:01:02	bb733c4c...
#2: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10*	02/19/2025 09:52:34	5f2674bc...
#3: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_halfdt_restart1*	02/19/2025 09:30:33	e15cdf0a...
#4: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_restart1*	02/19/2025 08:55:25	34b53db8...
#5: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 16:49:12	ccada4b5...
#6: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 14:05:45	917b7190...
#7: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR2_k3_useBCMap_ReInit5_restart1*	02/18/2025 13:52:09	04

In [ ]:
//sessions.Pick(0).Export().WithSupersampling(2).Do();

In [ ]:
var selection = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_2D"));
selection

#0: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10*	02/19/2025 09:52:34	5f2674bc...
#1: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_halfdt_restart1*	02/19/2025 09:30:33	e15cdf0a...
#2: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_restart1*	02/19/2025 08:55:25	34b53db8...
#3: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 16:49:12	ccada4b5...
#4: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 14:05:45	917b7190...
#5: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR2_k3_useBCMap_ReInit5_restart1*	02/18/2025 13:52:09	049ec74e...
#6: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x8AMR1_k3_useBCMap_ReInit5*	02/18/2025 12:04:23	6080af1f...
#7: DropletRebound_Gauthier	DropletReboundGauthier_2D_8x8AMR1_k3_useBCMap_ReInit5_restart1_noGravity*	02/18/2025 12:03:22	08839f51...
#8: DropletRebo

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions
//sessions.Pick(32).RestartedFrom

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.Pick(0).Delete(true)
//selection.Pick(0).Copy(databases.Pick(4))

In [ ]:
// // check for backup on userspace database
// var backupDB = OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound_Gauthier");
// List<ISessionInfo> noBackUpSessions = new List<ISessionInfo>();
// foreach(var sess in sessions) {
//     if(!backupDB.Sessions.Contains(sess)) {
//         noBackUpSessions.Add(sess);
//     }
// }
// noBackUpSessions

In [ ]:
//noBackUpSessions.MoveAll(backupDB);

In [ ]:
ISessionInfo restartSession;
if(restartRun == true) {
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x8x8AMR1_k3_ReI4_restart4")).Single();
    //restartSession = sessions.Where(sess => sess.Name.Contains("DropletReboundGauthier_8x12x8AMR1_k2_ReI4_restart2")).Single();
    restartSession = databases.Pick(0).Sessions.Pick(2);
}
restartSession

DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10*	02/19/2025 09:52:34	5f2674bc...

In [ ]:
// List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
// restartSessionList.Add(restartSession);
// Guid restartID = restartSession.RestartedFrom;
// while(restartID != Guid.Empty) {
//     var rSess = sessions.Where(sess => sess.ID == restartID).Single();
//     restartSessionList.Add(rSess);
//     restartID = rSess.RestartedFrom;
// }
// restartSessionList

In [ ]:
//restartSessionList.CopyAll(databases.Pick(1));
//restartSessionList.Pick(2).Timesteps.Skip(0).Export().WithSupersampling(2).Do();

In [ ]:
//restartSession.GetSessionDirectory()
//restartSession.Copy(databases.Pick(1));

In [ ]:
//restartSession.Timesteps.Count

In [ ]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(2).Do()

automated naming of restart sessions

In [ ]:
string restartName = (restartSession != null) ? String.Empty : "noRestart";

In [ ]:
if(restartSession != null) {
Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
procSIs.Push(restartSession);
var currSI = restartSession;
var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
while(!rSIs.IsNullOrEmpty()) {
    var rSI = rSIs.Single();
    procSIs.Push(rSI);
    currSI = rSI;
    rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
}
int restartNum = procSIs.Count;

string orgName = restartSession.Name;
if (restartNum == 1) {
    //restartName = orgName.Substring(0, orgName.Length - 10); // remoinvg _InitState
//} else if (restartNum == 1){
    restartName = orgName.Substring(0, orgName.Length) + "_restart1"; // + restartNum; 
} else {
    restartName = orgName.Substring(0, orgName.Length - 1) + restartNum; 
}
}
restartName

DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_restart1

In [ ]:
//restartName = "DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_Picard_restart1";

## Boundary Conditions

0: pressure outlet, 1: velocity inlet

In [ ]:
static int BC_tag2_top = 0;

## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [ ]:
static public Grid2D RotatingDiskSlice_CartesianCutOut(double radiusOP, double l_upstream, double l_downstream, double h_axial, int res_azimuthal, int res_axial) {

    double[] yNodes = GenericBlas.Linspace(-l_upstream, l_downstream, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid2D.Cartesian2DGrid(yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSlice2D_CartesianCutOut_{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");

    if (BC_tag2_top == 0)
        grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    if (BC_tag2_top == 1)
        grd.EdgeTagNames.Add(2, "velocity_inlet_top");
   
    grd.EdgeTagNames.Add(3, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(4, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.y.Abs() <= 1e-8)
            et = 1;
        if ((X.y - h_axial).Abs() <= 1e-8)
            et = 2;
        if ((X.x + (l_upstream)).Abs() <= 1e-8)
            et = 3;
        if ((X.x - (l_downstream)).Abs() <= 1e-8)
            et = 4;

        return et;
    });    

    return grd;
}

## Simulation setup

In [ ]:
double radiusOP = 90e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.2;  
double kinViscosity = 2.0e-5 / density; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double velOP = 38.0; 
double omega = velOP / radiusOP; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

0.00019867985355975657

Reynolds number at the point of injection

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

452.990066116245

Boundary layer (BL) thickness

In [ ]:
double zBL = 5.5;
double zBLstar = zBL * Lstar;
zBLstar

0.001092739194578661

In [ ]:
double radiusDrop = 0.91e-3;
double topDropPos = zBLstar + (2.0*radiusDrop);
topDropPos

0.002912739194578661

Height of the computational domain 

In [ ]:
double zTop = 20;
double zTopStar = zTop * Lstar;
zTopStar

0.003973597071195131

## Prescribed boundary conditions for the flow field 

The B.C. are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");

In [ ]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX2D();
vonKarmanHAM_velX.SetData(HAMcoeff_velV, omega, kinViscosity, radiusOP);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [ ]:
omega

422.22222222222223

In [ ]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 422.22222222222223; "  + 
    "double radiusOP = 90e-3; " + 
    "double velX = radiusOP * omega;" +
    "return velX; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [ ]:
// double velW_top = -0.88447342;
// double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
// velWstar_top

In [ ]:
// Formula VelocityZ_top = new Formula(
//     "VelZ",
//     false,
//     "double VelZ(double[] X) { return -0.07419586537108543; } "
// );

## Initial conditions for the injected droplet 

If not restarted from precomputed initial state with impact velocity for the droplet.

In [ ]:
radiusOP

0.09

In [ ]:
radiusDrop

0.00091

In [ ]:
double initHeight = zBLstar + radiusDrop;
initHeight

0.0020027391945786613

In [ ]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 0.09;" +
    "double radiusDrop = 0.91e-3;" +   
    "double initHeight = 1.5e-3;" + 
    "return Math.Sqrt(X[0].Pow2() + (X[1] - initHeight).Pow2()) - radiusDrop; } "
);

In [ ]:
Formula InitVelocity = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { return -0.23; } "
);

In [ ]:
Formula GravityValue = new Formula(
    "GravY",
    false,
    "double GravY(double[] X) { return -9.81; } " //-9.81; } "
);

## Control settings

In [ ]:
// set grid 
int res_global = 8;

double l_upstream = zTopStar * (2.0 / 2.0);
double l_downstream = zTopStar * (4.0 / 2.0);
double h_axial = 2.0 * zTopStar; 

int res_azimuthal = 3 * res_global;
int res_axial = 2 * res_global;

In [ ]:
double densityDrop = 960;
double sigma = 21e-3;

int AMRlevel_disk = 1;
int AMRlevel_drop = 2;
int AMRlevel_dropBL = 0;
int maxAMRlevel = (new int[] {AMRlevel_disk, AMRlevel_drop, AMRlevel_dropBL}).Max();
double hmin = (zTopStar / res_global) / (maxAMRlevel + 1);

int k = 3;

double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(densityDrop, density, sigma, hmin, k+1);
dt_sigma

1.6263393526180426E-05

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();


C.SetDGdegree(k);
C.FieldOptions.Add(VariableNames.GravityY, new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});


// physical parameters
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = 96e-3;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.IncludeConvection = true;

    
if (restartRun) {
    int RestartTimestepNumber = 289;
    C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, RestartTimestepNumber);
    C.ReInitTimestepIndex = RestartTimestepNumber;

} else {
    Grid2D grd = RotatingDiskSlice_CartesianCutOut(radiusOP, l_upstream, l_downstream, h_axial, res_azimuthal, res_axial);
    C.SetGrid(grd);

    // initial conditions
    C.AddInitialValue("VelocityX#B", vonKarmanHAM_velX);

    C.AddInitialValue("Phi", PhiFunc);
    C.AddInitialValue("VelocityY#A", InitVelocity);

}

C.AddInitialValue("GravityY#A", GravityValue);
C.AddInitialValue("GravityY#B", GravityValue);

// boundary conditions
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);

if (BC_tag2_top == 1) {
    C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
}

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#B", vonKarmanHAM_velX);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.ReInitPeriod = 10;

// C.SkipSolveAndEvaluateResidual = true;

C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

// C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();

//C.AgglomerationThreshold = 0.2;

// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.Timestepper_LevelSetHandling = LevelSetHandling.None;
// C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 1e-5;
C.NoOfTimesteps = 500;
    
{
    C.AdaptiveMeshRefinement = true;
    //C.activeAMRlevelIndicators.Add(new AMRwithGaussCheck() { maxRefinementLevel = AMRlevel_drop });
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
    //C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_dropBL });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_disk });
    C.AMR_startUpSweeps = AMRlevel_drop;
}

// C.PostprocessingModules.Add(new EnergyLogging());
C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers,BoSSS.Foundation.ConstrainedDGprojection";
    
if (restartRun) 
    C.SessionName = restartName;
else {
    C.SessionName = $"DropletReboundGauthier_2D_{res_azimuthal}x{res_axial}AMR{maxAMRlevel}_k{k}_useBCMap_ReInit{C.ReInitPeriod}";
}
    
    
Controls.Add(C);

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_Picard_restart1


In [ ]:
restartSession

DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10*	02/19/2025 09:52:34	5f2674bc...

In [ ]:
myBatch.Name

FDY-WindowsHPC

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 2;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    oneJob.Activate(myBatch); 
}

In [ ]:
wmg.Sessions

#0: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_Picard_restart1*	02/20/2025 10:25:48	ad20f851...
#1: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10_LineSearch_restart1*	02/20/2025 10:07:39	aa479395...
#2: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5_LineSearch_restart1*	02/20/2025 10:01:02	bb733c4c...
#3: DropletRebound_Gauthier	DropletReboundGauthier_2D_48x32AMR2_k3_useBCMap_ReInit10*	02/19/2025 09:52:34	5f2674bc...
#4: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_halfdt_restart1*	02/19/2025 09:30:33	e15cdf0a...
#5: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5_restart1*	02/19/2025 08:55:25	34b53db8...
#6: DropletRebound_Gauthier	DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 16:49:12	ccada4b5...
#7: DropletRebound_Gauthier	DropletReboundGauthier_2D_16x16AMR1_k3_useBCMap_ReInit5*	02/18/2025 1

In [ ]:
wmg.Sessions.Pick(2).Export().WithSupersampling(2).Do()

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\DropletRebound_Gauthier__DropletReboundGauthier_2D_32x16AMR1_k3_useBCMap_ReInit5_LineSearch_restart1__bb733c4c-172b-4e7a-8997-ff8652ba32cb

In [ ]:
//wmg.Sessions.Pick(0).Delete(true)

## Wait for Completion and Check Job Status

In [ ]:
double hrsToWait = 0.25;

In [ ]:
wmg.BlockUntilAllJobsTerminate(hrsToWait*3600);

unable to determine job status - unknown
unable to determine job status - unknown
Timeout.


In [ ]:
var JobStat = Controls.Select(ctrl => ctrl.GetJob().Status).ToArray();
JobStat

index,value
0,InProgress


In [ ]:
NUnit.Framework.Assert.Zero(JobStat.Where(jS => (jS == BoSSS.Application.BoSSSpad.JobStatus.FailedOrCanceled)).Count(), "Some Jobs Failed");